# <span style="color:#00bfff;">Python Asíncrono</span>

## Un resumen sobre la programación asíncrona en Python

La programación **asíncrona** permite que un programa siga trabajando mientras espera algo "lento" (una respuesta de red, leer un archivo, esperar a una API de un LLM), en lugar de quedarse bloqueado sin hacer nada. En Python esto se logra con el módulo `asyncio`, las funciones `async def` y la palabra clave `await`.

A diferencia del *multithreading*, `asyncio` usa un único hilo con un **bucle de eventos** (*event loop*) que va alternando entre tareas cada vez que una de ellas queda en espera. Es ideal para tareas de I/O (red, disco, APIs), no para cálculos pesados de CPU.

> <span style="color:#ff7800;">_**Nota importante:**_</span> _En varias celdas de este notebook verá `await` escrito directamente en una celda, fuera de cualquier función. Esto **solo funciona en Jupyter/IPython**, porque estos entornos ya mantienen un bucle de eventos corriendo por detrás. En un script `.py` normal, escribir `await` fuera de una función `async def` produce un `SyntaxError`; ahí se debe envolver el código en una función `async def main(): ...` y ejecutarla con `asyncio.run(main())`._

In [1]:
# Definamos una función asíncrona

import asyncio

async def do_some_work():
    print("Iniciando trabajo")
    await asyncio.sleep(1)
    print("Trabajo completado")

Con `async def` definimos una **corrutina**: una función especial que puede "pausarse" en cada `await` sin bloquear el resto del programa. `asyncio.sleep(1)` simula una espera de 1 segundo (como si fuera una llamada de red), cediendo el control al bucle de eventos mientras espera.

¿Qué pasará si llamamos a `do_some_work()` como si fuera una función normal, sin `await`?

In [2]:
# ¿Qué hará esto?

do_some_work()

<coroutine object do_some_work at 0x0000024334FA6A40>

No imprime "Iniciando trabajo" ni "Trabajo completado". Llamar a una función `async def` sin `await` **no la ejecuta**: solo crea un objeto `coroutine` (por eso el resultado es algo como `<coroutine object do_some_work at 0x...>`). Para que el código dentro realmente se ejecute, hay que usar `await`.

In [3]:
# ¡OK, intentémoslo de nuevo!

await do_some_work()

Iniciando trabajo
Trabajo completado


Ahora sí se ejecuta correctamente. Veamos un error común: llamar a varias corrutinas dentro de otra función `async` **sin `await`** en cada una.

In [ ]:
# ¿Qué está mal con esto?

async def do_a_lot_of_work():
    do_some_work()
    do_some_work()
    do_some_work()

await do_a_lot_of_work()

Este código se ejecuta casi instantáneamente y no imprime nada de `do_some_work()`, además de mostrar advertencias `RuntimeWarning: coroutine 'do_some_work' was never awaited`. Es el mismo error que antes, repetido tres veces: cada llamada crea una corrutina que nunca se ejecuta porque le falta `await`. Corrijámoslo.

In [5]:
# ¡Advertencia interesante! Corrijámoslo

async def do_a_lot_of_work():
    await do_some_work()
    await do_some_work()
    await do_some_work()

await do_a_lot_of_work()

Iniciando trabajo
Trabajo completado
Iniciando trabajo
Trabajo completado
Iniciando trabajo
Trabajo completado


Ahora sí se ejecutan las tres llamadas, pero **una tras otra**: como cada `await` espera a que termine la anterior antes de seguir, el total tarda ~3 segundos (3 × 1 segundo). Esto sigue siendo secuencial, no concurrente. Para que las tres corran *al mismo tiempo* usamos `asyncio.gather()`.

In [6]:
# Y ahora hagámoslo en paralelo
# Es importante reconocer que esto no es "multithreading" de la manera a la que podría estar acostumbrado.
# La librería asyncio se ejecuta en un solo hilo, pero utiliza un bucle de eventos para cambiar entre tareas mientras una está en espera.

async def do_a_lot_of_work_in_parallel():
    await asyncio.gather(do_some_work(), do_some_work(), do_some_work())

await do_a_lot_of_work_in_parallel()

Iniciando trabajo
Iniciando trabajo
Iniciando trabajo
Trabajo completado
Trabajo completado
Trabajo completado


Ahora las tres tareas inician casi al mismo tiempo ("Iniciando trabajo" se imprime 3 veces seguidas) y el total tarda ~1 segundo en vez de 3: mientras una está en `await asyncio.sleep(1)`, el bucle de eventos aprovecha para avanzar las otras dos. Esta es la base de por qué `asyncio` es útil para hacer varias llamadas de red o a APIs (por ejemplo, a un LLM) en paralelo sin usar hilos.

Fuera de Jupyter, este mismo ejemplo se estructura como:

```python
import asyncio

async def main():
    await asyncio.gather(do_some_work(), do_some_work(), do_some_work())

if __name__ == "__main__":
    asyncio.run(main())
```